In [4]:
import rasterio
import geopandas as gpd
import numpy as np
from rasterio.mask import mask

chirps_2023_path = "../data/raw/chirps_india/chirps-v2.0.2023.tif"
boundary_path = "../data/raw/boundaries/ne_10m_admin_0_countries.shp"

# Load India boundary
countries = gpd.read_file(boundary_path)
india = countries[countries["NAME"] == "India"].geometry.iloc[0]

# Open CHIRPS
with rasterio.open(chirps_2023_path) as src:

    # Make sure India boundary uses the same CRS
    india = gpd.GeoSeries([india], crs="EPSG:4326").to_crs(src.crs).iloc[0]

    # Extract pixels inside India
    rainfall, transform = mask(
        src,
        [india.__geo_interface__],
        crop=True,
        filled=False
    )

    rainfall = rainfall[0]

    # Latitude of each raster row
    rows = np.arange(rainfall.shape[0])
    latitudes = transform.f + (rows + 0.5) * transform.e

    # Latitude-area weighting
    weights = np.cos(np.deg2rad(latitudes))[:, np.newaxis]

    # Calculate weighted mean
    valid = ~rainfall.mask
    rainfall_values = rainfall.filled(np.nan)

    weighted_mean = np.nansum(
        rainfall_values * weights
    ) / np.nansum(
        weights * valid
    )

print(f"CHIRPS India annual precipitation for 2023: {weighted_mean:.2f} mm")

CHIRPS India annual precipitation for 2023: 1094.75 mm


In [5]:
import xarray as xr
import geopandas as gpd
import numpy as np
from shapely import contains_xy

# Paths
era5_path = "../data/raw/era5_india/data_stream-moda.nc"
boundary_path = "../data/raw/boundaries/ne_10m_admin_0_countries.shp"

# Load ERA5
ds = xr.open_dataset(era5_path)

print("Temperature units:", ds["t2m"].attrs.get("units"))
print("Precipitation units:", ds["tp"].attrs.get("units"))
print("Precipitation description:", ds["tp"].attrs.get("long_name"))

# Load India boundary
countries = gpd.read_file(boundary_path)
india = countries[countries["NAME"] == "India"].geometry.iloc[0]

# Create India grid mask
lon, lat = np.meshgrid(ds.longitude.values, ds.latitude.values)
india_mask = contains_xy(india, lon, lat)

# Select only 2023
tp_2023 = ds["tp"].sel(valid_time=slice("2023-01-01", "2023-12-01"))

# Apply India mask
tp_2023_india = tp_2023.where(india_mask)

# Area weighting
weights = np.cos(np.deg2rad(ds.latitude))

# Area-weighted monthly precipitation
monthly_tp = (
    tp_2023_india.weighted(weights)
    .mean(dim=("latitude", "longitude"))
)

# Convert m/day → mm/month
days = tp_2023["valid_time"].dt.days_in_month
monthly_rainfall_mm = monthly_tp * 1000 * days

# Annual total
annual_rainfall_2023 = monthly_rainfall_mm.sum().item()

print("\nERA5-Land 2023 monthly precipitation (mm):")
for date, value in zip(
    tp_2023.valid_time.values,
    monthly_rainfall_mm.values
):
    print(str(date)[:7], f"{value:.2f}")

print(f"\nERA5-Land India 2023 annual precipitation: {annual_rainfall_2023:.2f} mm")

Temperature units: K
Precipitation units: m
Precipitation description: Total precipitation

ERA5-Land 2023 monthly precipitation (mm):
2023-01 8.77
2023-02 6.24
2023-03 24.92
2023-04 24.01
2023-05 36.61
2023-06 89.77
2023-07 179.73
2023-08 98.46
2023-09 114.16
2023-10 29.18
2023-11 24.09
2023-12 15.58

ERA5-Land India 2023 annual precipitation: 651.51 mm


In [6]:
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd

# --------------------------------------------------
# Paths
# --------------------------------------------------

era5_india_path = "../data/raw/era5_india/data_stream-moda.nc"
era5_usa_path = "../data/raw/era5_usa/data_stream-moda.nc"

boundary_path = "../data/raw/boundaries/ne_10m_admin_0_countries.shp"


# --------------------------------------------------
# Load country boundaries
# --------------------------------------------------

countries = gpd.read_file(boundary_path)


# --------------------------------------------------
# Function: calculate annual ERA5 precipitation
# --------------------------------------------------

def calculate_annual_precipitation(nc_path, country_name):
    
    ds = xr.open_dataset(nc_path)

    # Find country boundary
    country = countries[countries["ADMIN"] == country_name].geometry.iloc[0]

    # Create longitude-latitude grid
    lon, lat = np.meshgrid(
        ds.longitude.values,
        ds.latitude.values
    )

    # Country mask
    from shapely import contains_xy

    country_mask = contains_xy(country, lon, lat)

    # Select precipitation
    tp = ds["tp"].where(country_mask)

    # Latitude-area weighting
    weights = np.cos(np.deg2rad(ds.latitude))

    # Area-weighted monthly precipitation
    monthly_tp = (
        tp.weighted(weights)
        .mean(dim=("latitude", "longitude"))
    )

    # Convert ERA5 tp:
    # m/day → mm/month
    days = ds["valid_time"].dt.days_in_month

    monthly_rainfall_mm = monthly_tp * 1000 * days

    # Convert to DataFrame
    df = pd.DataFrame({
        "date": pd.to_datetime(ds["valid_time"].values),
        "precipitation_mm": monthly_rainfall_mm.values
    })

    df["year"] = df["date"].dt.year

    # Annual total
    annual = (
        df.groupby("year")["precipitation_mm"]
        .sum()
        .reset_index()
    )

    annual["country"] = country_name

    ds.close()

    return annual


# --------------------------------------------------
# INDIA: check 2021–2023
# --------------------------------------------------

india_annual = calculate_annual_precipitation(
    era5_india_path,
    "India"
)

print("INDIA — 2021 to 2023")
print(
    india_annual[
        india_annual["year"].between(2021, 2023)
    ][["year", "precipitation_mm"]]
)


# --------------------------------------------------
# USA: calculate 2000–2023
# --------------------------------------------------

usa_annual = calculate_annual_precipitation(
    era5_usa_path,
    "United States of America"
)

print("\nUSA — 2000 to 2023")
print(
    usa_annual[
        ["year", "precipitation_mm"]
    ].to_string(index=False)
)

INDIA — 2021 to 2023
    year  precipitation_mm
21  2021       1298.552714
22  2022       1181.258564
23  2023        651.514299

USA — 2000 to 2023
 year  precipitation_mm
 2000        742.390627
 2001        751.077695
 2002        754.488099
 2003        783.582540
 2004        849.164202
 2005        767.795952
 2006        765.194581
 2007        738.145327
 2008        807.253400
 2009        838.357096
 2010        801.013957
 2011        774.317818
 2012        695.976205
 2013        786.077764
 2014        777.850863
 2015        853.920655
 2016        811.040342
 2017        859.588144
 2018        916.686994
 2019        927.304706
 2020        789.631812
 2021        811.931607
 2022        643.855048
 2023        383.212168


In [7]:
import rasterio

chirps_path = "../data/raw/chirps_india/chirps-v2.0.1981-2023.43yrs.tif"

with rasterio.open(chirps_path) as src:
    print("Number of bands:", src.count)
    print("Width:", src.width)
    print("Height:", src.height)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes[0])
    print("NoData:", src.nodata)
    print("Band descriptions:", src.descriptions)
    print("Band 1 tags:", src.tags(1))
    print("Dataset tags:", src.tags())

Number of bands: 1
Width: 7200
Height: 2000
CRS: EPSG:4326
Bounds: BoundingBox(left=-180.0, bottom=-50.00000149011612, right=180.00000536441803, top=50.0)
Data type: float32
NoData: None
Band descriptions: (None,)
Band 1 tags: {}
Dataset tags: {'TIFFTAG_DOCUMENTNAME': '/home/CHIRPS/annual/v2.0/chirps-v2.0.1981-2023.43yrs.tif', 'TIFFTAG_IMAGEDESCRIPTION': 'IDL TIFF file', 'TIFFTAG_SOFTWARE': 'IDL 8.9.0, L3Harris Geospatial Solutions, Inc.', 'TIFFTAG_DATETIME': '2024:01:11 14:54:30', 'TIFFTAG_XRESOLUTION': '100', 'TIFFTAG_YRESOLUTION': '100', 'TIFFTAG_RESOLUTIONUNIT': '2 (pixels/inch)', 'AREA_OR_POINT': 'Area'}


In [10]:
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely import contains_xy


# ============================================================
# 1. File paths
# ============================================================

boundary_path = "../data/raw/boundaries/ne_10m_admin_0_countries.shp"

chirps_files = {
    2021: "../data/raw/chirps_india/chirps-v2.0.2021.tif",
    2022: "../data/raw/chirps_india/chirps-v2.0.2022.tif",
    2023: "../data/raw/chirps_india/chirps-v2.0.2023.tif"
}


# ============================================================
# 2. Load India boundary
# ============================================================

countries = gpd.read_file(boundary_path)

india = countries[
    countries["ADMIN"] == "India"
].geometry.iloc[0]


# ============================================================
# 3. Function to calculate India-wide CHIRPS rainfall
# ============================================================

def calculate_chirps_india_rainfall(tif_path):

    with rasterio.open(tif_path) as src:

        # Read rainfall raster
        rainfall = src.read(1)

        # Create row and column indices
        rows, cols = np.indices(rainfall.shape)

        # Get longitude and latitude of every pixel
        lon, lat = rasterio.transform.xy(
            src.transform,
            rows,
            cols,
            offset="center"
        )

        # Convert to NumPy arrays
        # and make them the same shape as rainfall
        lon = np.array(lon).reshape(rainfall.shape)
        lat = np.array(lat).reshape(rainfall.shape)

        # ----------------------------------------------------
        # Create India mask
        # ----------------------------------------------------

        india_mask = contains_xy(
            india,
            lon,
            lat
        )

        # ----------------------------------------------------
        # Keep only valid pixels inside India
        # ----------------------------------------------------

        valid = (
            india_mask &
            np.isfinite(rainfall)
        )

        # ----------------------------------------------------
        # Latitude weighting
        # ----------------------------------------------------

        weights = np.cos(
            np.deg2rad(lat)
        )

        # ----------------------------------------------------
        # Area-weighted mean rainfall
        # ----------------------------------------------------

        india_rainfall = (
            np.sum(
                rainfall[valid] *
                weights[valid]
            )
            /
            np.sum(
                weights[valid]
            )
        )

    return india_rainfall


# ============================================================
# 4. Calculate rainfall for 2021, 2022 and 2023
# ============================================================

results = []

for year, path in chirps_files.items():

    rainfall = calculate_chirps_india_rainfall(path)

    results.append({
        "year": year,
        "CHIRPS_precipitation_mm": rainfall
    })


# ============================================================
# 5. Create results table
# ============================================================

chirps_india = pd.DataFrame(results)


# ============================================================
# 6. Display results
# ============================================================

print("CHIRPS — INDIA")
print(
    chirps_india.to_string(index=False)
)

CHIRPS — INDIA
 year  CHIRPS_precipitation_mm
 2021              1269.509063
 2022              1280.868780
 2023              1094.751932


In [11]:
# Compare ERA5 and CHIRPS precipitation

comparison = pd.DataFrame({
    "year": [2021, 2022, 2023],
    "ERA5_precipitation_mm": [1298.552714, 1181.285564, 651.514299],
    "CHIRPS_precipitation_mm": [
        1269.509063,
        1280.868780,
        1094.751932
    ]
})

comparison["ERA5_CHIRPS_ratio"] = (
    comparison["ERA5_precipitation_mm"]
    / comparison["CHIRPS_precipitation_mm"]
)

comparison["difference_percent"] = (
    (comparison["ERA5_precipitation_mm"]
     - comparison["CHIRPS_precipitation_mm"])
    / comparison["CHIRPS_precipitation_mm"]
    * 100
)

print(comparison.to_string(index=False))

 year  ERA5_precipitation_mm  CHIRPS_precipitation_mm  ERA5_CHIRPS_ratio  difference_percent
 2021            1298.552714              1269.509063           1.022878            2.287786
 2022            1181.285564              1280.868780           0.922253           -7.774662
 2023             651.514299              1094.751932           0.595125          -40.487495


In [12]:
import xarray as xr
import numpy as np
import pandas as pd

era5_india_path = "../data/raw/era5_india/data_stream-moda.nc"

ds = xr.open_dataset(era5_india_path)

# India boundary mask
lon, lat = np.meshgrid(
    ds.longitude.values,
    ds.latitude.values
)

india_mask = contains_xy(
    india,
    lon,
    lat
)

# Extract precipitation over India
tp = ds["tp"].where(india_mask)

# Latitude weighting
weights = np.cos(
    np.deg2rad(ds.latitude)
)

# Area-weighted monthly precipitation
monthly_tp = tp.weighted(weights).mean(
    dim=("latitude", "longitude")
)

# Convert ERA5 precipitation to mm/month
days = ds["valid_time"].dt.days_in_month

monthly_rainfall_mm = (
    monthly_tp * 1000 * days
)

# Create table for 2023
india_2023 = pd.DataFrame({
    "date": pd.to_datetime(
        ds["valid_time"].values
    ),
    "precipitation_mm": monthly_rainfall_mm.values
})

india_2023["year"] = india_2023["date"].dt.year

india_2023 = india_2023[
    india_2023["year"] == 2023
].copy()

print("ERA5 INDIA — MONTHLY PRECIPITATION 2023")
print(
    india_2023[
        ["date", "precipitation_mm"]
    ].to_string(index=False)
)

print(
    "\n2023 total:",
    india_2023["precipitation_mm"].sum(),
    "mm"
)

ERA5 INDIA — MONTHLY PRECIPITATION 2023
      date  precipitation_mm
2023-01-01          8.770142
2023-02-01          6.241601
2023-03-01         24.915019
2023-04-01         24.014144
2023-05-01         36.611677
2023-06-01         89.770729
2023-07-01        179.733288
2023-08-01         98.457986
2023-09-01        114.158117
2023-10-01         29.177124
2023-11-01         24.086482
2023-12-01         15.577990

2023 total: 651.5142991847648 mm
